In [ ]:
import os
from os import path
import pickle
import numpy as np
from glob import glob
import pandas as pd

from macaquethings.plotting.default_styles import *
from macaquethings.plotting.util import legend_without_duplicate_labels
from macaquethings.rdm_util import get_rdm_design_sort_indices

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.ticker as tkr


figure_style()  # set consistent plotting defaults for all figs

# Select Data

In [ ]:
# if true, plot source signal and subspace
# if false, plot target signal and predictions
plot_in_source_space = False 
savedir = path.join("rdm_dnn_corr_subspace_panels", "source" if plot_in_source_space else "target")
os.makedirs(savedir, exist_ok=True)

run_group_name = "monkeyF_v4_it"

rundirs_F_v4_it = [
    "monkeyF_v4_target_array_9_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v4_target_array_10_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v4_target_array_11_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v4_target_array_12_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v4_target_array_13_stride_2_ridgecv_lfp_threefold",
    ]

rundirs_N_v4_it = [
    "monkeyN_v4_target_array_13_stride_2_ridgecv_lfp_threefold",
    "monkeyN_v4_target_array_14_stride_2_ridgecv_lfp_threefold",
    "monkeyN_v4_target_array_15_stride_2_ridgecv_lfp_threefold",
    "monkeyN_v4_target_array_16_stride_2_ridgecv_lfp_threefold",
    ]

rundirs_F_v1_it = [
    "monkeyF_v1_target_array_9_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v1_target_array_10_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v1_target_array_11_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v1_target_array_12_stride_2_ridgecv_lfp_threefold",
    "monkeyF_v1_target_array_13_stride_2_ridgecv_lfp_threefold",
    ]

rdm_corr_fname = "rdm_corrs_rdms-resnet18-metric_cosine-normalization_None-pca_1000.pkl.pkl"

# time, delay pairs
subspaces_F_v4_it = [
    (74, 28),
    (120, 40)
]

subspaces_N_v4_it = [
    (62, 24),
    (112, 36)
]

subspaces_F_v1_it = [
    (58, 26),
    (72, 28)
]

group2data = {
    "monkeyF_v4_it": (rundirs_F_v4_it, subspaces_F_v4_it),
    "monkeyF_v1_it": (rundirs_F_v1_it, subspaces_F_v1_it),
    "monkeyN_v4_it": (rundirs_N_v4_it, subspaces_N_v4_it),
    
}

rundirs, subspaces = group2data[run_group_name]

# Collect DNN Corr Results

In [ ]:
data_per_run = []

for rundir in rundirs:
    try:
        analysis_dir = path.join("..", "..", "results", "inter_area", rundir, "analysis")
        subspace_dirs = [
            path.join(analysis_dir, f"subspace_t-{t}_d-{d}") for t,d in subspaces
        ]
        dnn_corrs_per_subspace = []
        
        # ----- collect source data dnn corrs
        # get source timecourse data first, same for all subspaces so pick from first folder
        subdir = subspace_dirs[0]
        with open(path.join(subdir, rdm_corr_fname), 'rb') as f:
            corr_data = pickle.load(f)
        
        # get keys and indices
        model_keys = corr_data["model_keys"]
        layer_indices = corr_data["layer_indices"]
        time = corr_data['time']

        if plot_in_source_space:
            dnn_corrs_per_subspace.append(corr_data['rdm_corrs_source'])
        else:
            dnn_corrs_per_subspace.append(corr_data['rdm_corrs_target'])
            
        
        for subdir in subspace_dirs:
            with open(path.join(subdir, rdm_corr_fname), 'rb') as f:
                corr_data = pickle.load(f)
        
            # assert the ordering is the same so we can safely concatenate
            mk = corr_data["model_keys"]
            li = corr_data["layer_indices"]
            t = corr_data["time"]
        
            assert (np.array(model_keys) == np.array(mk)).all(), 'inconsistent ordering of model keys.'
            assert (np.array(layer_indices) == np.array(li)).all(), 'inconsistent ordering of layer indices.'
            assert (time == t).all(), "inconsistent time vectors"

            if plot_in_source_space:
                dnn_corrs_per_subspace.append(corr_data['rdm_corrs_sub'])  
            else:
                dnn_corrs_per_subspace.append(corr_data['rdm_corrs_pred'])  
    
        data_per_run.append((model_keys, layer_indices, time, dnn_corrs_per_subspace))
    except Exception as e:
        print("failed to load data for dir", rundir)
        print(e)
        raise e


# Plotting

In [ ]:
"""
colors = np.array(
    sns.color_palette("Blues", len(model_keys) + 1)[1:]
)  # exclude first color, since it is too faint
"""
colors = np.array(
    sns.color_palette("magma", len(model_keys))
)

# sort colors by  layer position
layer_idx = np.argsort(layer_indices)
colors = colors[layer_idx]

# cmap for colorbar
cmap = ListedColormap(colors)
bounds = np.arange(len(colors) + 1)
norm = BoundaryNorm(bounds, cmap.N)



In [ ]:
xlims = (0, 300)


def plot_dnn_timecourse_panels(model_keys, layer_indices, time, dnn_corrs_per_subspace, subspaces, plot_in_source_space, xlims=None, ylims=None, separate_figs=False):

    if not separate_figs:
        fig, axs = plt.subplots(1, 3, figsize=(FULL_PANEL_SIZE[0], 1.5), sharey=True)
        figs = [fig]

    else:
        figs = []
        axs = []
        for _ in range(3):
            fig, ax = plt.subplots(1, 1, figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))
            figs.append(fig)
            axs.append(ax)
    
    for i, (ax, dnn_corrs) in enumerate(zip(axs, dnn_corrs_per_subspace)):
        ax.set_ylabel("rdm corr.")
        if plot_in_source_space:
            ax.set_title("source signal")
        else:
            ax.set_title("target signal")
        if i > 0:
            t, d = subspaces[i - 1]
            ax.set_title(f"Subspace t={t}ms d={d}ms")
    
        ax.set_xlabel("time (ms)")
        dnn_corrs_avg = dnn_corrs.mean(axis=0)  # avg. over folds
        
        for idx, corr in zip(layer_idx, dnn_corrs_avg.T):
            if i > 0:  # this is a subspace trajectory - if we are in target space we account for the delay in predictions
                xs = time if plot_in_source_space else time + d  # account for delay in predictions if we are looking at target time
            else:  # this is a raw time course, there is never any delay
                xs = time
                            
            ax.plot(xs, corr, color=colors[idx])

        if i > 0:  # we are looking at a subspace signal that has a time and delay for which the model was fit
            tmarker = (t - d) if plot_in_source_space else t  # indicate the time point used to fit the model. In source time, we account for a delay
            for ax in axs:  # in all panels, indicate the times the models were fit on. This is either a line for the target time if we are in target space, or a line for t - d, the time we read from the source region
                ax.axvline(tmarker, color='gray', alpha=.5, linewidth=.5)

        if not xlims is None:
            for ax in axs:
                ax.set_xlim(xlims)

        if not ylims is None:
            for ax in axs:
                ax.set_ylim(ylims)
 
    return figs, axs

        
            

for run_idx, (model_keys, layer_indices, time, dnn_corrs_per_subspace) in enumerate(data_per_run):
    plot_dnn_timecourse_panels(model_keys, layer_indices, time, dnn_corrs_per_subspace, subspaces, plot_in_source_space, xlims=xlims)
    plt.suptitle(rundirs[run_idx].split("/")[-1])
    plt.tight_layout()

    fpath = path.join(
        savedir,
        rundirs[run_idx][:-4] + "_dnn_corrs_subspaces.svg"
    )
    plt.savefig(fpath)


# Plot avg. over target arrays

In [ ]:
ylims = (-.02, .35)

model_keys = np.array([d[0] for d in data_per_run])
layer_indices = np.array([d[1] for d in data_per_run])
time = np.array([d[2] for d in data_per_run])
dnn_corrs_per_subspace = np.array([d[3] for d in data_per_run])

print(model_keys.shape)
print(layer_indices.shape)
print(time.shape)
print(dnn_corrs_per_subspace.shape)

# assert sorting is the same
assert ((layer_indices - layer_indices[0][None, :]) == 0).all(), 'inconsistent sorting'
assert ((time - time[0][None, :]) == 0).all(), 'inconsistent sorting'

# same for all
time = time[0]
layer_indices = layer_indices[0]
model_keys = model_keys[0]

# avg over runs
dnn_corrs_per_subspace = dnn_corrs_per_subspace.mean(axis=0)

figs, axs = plot_dnn_timecourse_panels(model_keys, layer_indices, time, dnn_corrs_per_subspace, subspaces, plot_in_source_space, xlims=xlims, ylims=ylims, separate_figs=True)


for i, fig in enumerate(figs):
    fpath = path.join(
        savedir,
        run_group_name + f"_avg_dnn_corrs_subspaces_panel_{i+1}.svg"
    )
    fig.savefig(fpath)
    print(fpath)


# Plot Difference to Original Signal

In [ ]:
def align_target_trajectories(trajectories, delays, time):
    """
    a set of trajectories (can be multivariate) to align in time
    time is the shared time vector for all trajectories, not accounting for delays
    delays is the positive delay for each trajectory

    shifts all trajectories according to delays, and returns the truncated time series
    for which data is available for all possible delays (this will be bounded by the longest delay)

    trajectories has a set of trajectories stacked along the first axis
    the second axis must be time
    """
    # get first and last time point
    tmin, tmax = time.min(), time.max()

    print(f'min time {tmin} max time {tmax}')

    largest_delay = np.max(delays)  # largest delay

    print('delays', delays)
    
    min_shared_time = tmin + largest_delay  # first time point that is available for all delays
    max_shared_time = tmax

    print(f"first shared time point: {min_shared_time}")
    print(f"last shared time point: {max_shared_time}")
    

    n_shared_time = (time >= min_shared_time).sum()

    aligned_time = time[(time >= min_shared_time) & (time <= max_shared_time)]

    shifted_trajectories = []
    for trajectory, delay in zip(trajectories, delays):
        tshift = time + delay
        tmask = (tshift >= min_shared_time) & (tshift <= max_shared_time)
        shifted_trajectories.append(trajectory[tmask])

    for st in shifted_trajectories:
        print(st.shape)
        
    return aligned_time, np.array(shifted_trajectories)
    

# avg over folds as well
dnn_corr_avg_arrays_avg_folds = dnn_corrs_per_subspace.mean(axis=1)

# if we are working in target space, align trajectories
if not plot_in_source_space:
    aligned_time, dnn_corrs_avg_aligned = align_target_trajectories(dnn_corr_avg_arrays_avg_folds, [0] + [d for t, d in subspaces], time)
else:
    aligned_time, dnn_corrs_avg_aligned = align_target_trajectories(dnn_corr_avg_arrays_avg_folds, [0] + [0 for t, d in subspaces], time)    # no alignment needed in v4 time

# get diff with "full" signal
dnn_corr_diffs = (dnn_corrs_avg_aligned - dnn_corrs_avg_aligned[0][None, :, :])
print(dnn_corr_diffs.shape)


plt.figure(figsize=(FULL_WIDTH, FULL_WIDTH / 5))
plt.subplot(131)
for i in range(dnn_corr_diffs.shape[-1]):
    plt.plot(aligned_time, dnn_corr_diffs[0, :, i], color=colors[i])

plt.subplot(132)
for i in range(dnn_corr_diffs.shape[-1]):
    plt.plot(aligned_time, dnn_corr_diffs[1, :, i], color=colors[i])

plt.subplot(133)
for i in range(dnn_corr_diffs.shape[-1]):
    plt.plot(aligned_time, dnn_corr_diffs[2, :, i], color=colors[i])


# Contast Subspaces directly

In [ ]:
if plot_in_source_space:
    markers = [t - d for t,d in subspaces]
else:
    markers = [t for t, _ in subspaces]

In [ ]:
dnn_corr_sub_contrast = dnn_corrs_avg_aligned[1] - dnn_corrs_avg_aligned[2]
print(dnn_corr_sub_contrast.shape)


fig = plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))
for i, layer in enumerate(model_keys):
    plt.plot(aligned_time, dnn_corr_sub_contrast[:, i], label=layer, color=colors[i])
plt.axhline(0., color='k', linestyle='--', linewidth=.5)

for marker in markers:
    plt.axvline(marker, color='gray', alpha=.5, linewidth=.5)
plt.xlabel('time (ms')
plt.ylabel('dnn corr. early - late')
plt.xlim(xlims)

fpath = path.join(
    savedir,
    run_group_name + f"_avg_dnn_corrs_subspaces_contrast.svg"
)
fig.savefig(fpath)
print(fpath)



# Plot a colorbar over layers

In [ ]:
plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))
ax = plt.gca()
cb = plt.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax,
    shrink=1.4
)
cb.ax.minorticks_off()
cb.set_ticks(np.arange(len(model_keys)) + 0.5)  # center each tick
cb.set_ticklabels(model_keys)
cb.ax.tick_params(labelsize=5)
ax.axis('off')
plt.savefig(
    path.join(savedir, 'cbar_dnn_labels.svg')
)

# Plot RDMs in all subspaces for a selected time, delay pair

In [ ]:
from scipy.stats import rankdata
from scipy.spatial.distance import squareform

plt.close('all')

def get_subspace_rdms(rundir, subspace):
    analysis_dir = path.join("..", "..", "results", "inter_area", rundir, "analysis")
    tsub, dsub = subspace
    subdir = path.join(analysis_dir, f"subspace_t-{tsub}_d-{dsub}")
    
    # ----- collect source data dnn corrs
    # get source timecourse data first, same for all subspaces so pick from first folder
    with open(path.join(subdir, 'rdms.pkl'), 'rb') as f:
        sub_data = pickle.load(f)
    print(sub_data.keys())

    t_source = tsub - dsub
    t_target = tsub

    # get indices
    ts = sub_data["time"]

    tidx_source = np.where(t_source == ts)[0][0]
    tidx_target = np.where(t_target == ts)[0][0]
    
    # full source rdm
    source_rdm_full = sub_data['rdms_source'][:, tidx_source]
    source_rdm_sub = sub_data['rdms_source_readout_subspace'][:, tidx_source]
    target_rdm_pred = sub_data['rdms_target_prediction'][:, tidx_source]
    target_rdm_full = sub_data['rdms_target'][:, tidx_target]  # note that the index here accounts for the delay

    del sub_data  # make sure memory gets freed immediately

    return source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full


def plot_all_rdms_for_subspace_model(source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full, labels="filenames", rank=True):
    sort_idx = get_rdm_design_sort_indices('../..', return_values=False, reduce_to_column=labels)
    fig, axs = plt.subplots(1, 4, figsize=FULL_PANEL_SIZE)
    
    rdm_source = source_rdm_full.mean(axis=0)  # avg over folds
    if rank:
        rdm_source = rankdata(rdm_source)
    
    axs[0].imshow(squareform(rdm_source)[sort_idx][:, sort_idx])
    axs[0].set_title("source")
    
    rdm_source_sub = source_rdm_sub.mean(axis=0)  # avg over folds
    if rank:
        rdm_source_sub = rankdata(rdm_source_sub)
    axs[1].imshow(squareform(rdm_source_sub)[sort_idx][:, sort_idx])
    axs[1].set_title("subspace")

    rdm_pred = target_rdm_pred.mean(axis=0)  # avg over folds
    if rank:
        rdm_pred = rankdata(rdm_pred)
    axs[2].imshow(squareform(rdm_pred)[sort_idx][:, sort_idx])
    axs[2].set_title("prediction")

    rdm_target = target_rdm_full.mean(axis=0)  # avg over folds
    if rank:
        rdm_target = rankdata(rdm_target)
    axs[3].imshow(squareform(rdm_target)[sort_idx][:, sort_idx])
    axs[3].set_title("target")

    for ax in axs:
        ax.axis('off')

    return fig, axs
        


# Early vs Late RDMs in Subspace / Prediction / Target signal

# Subspaces

In [ ]:
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

early_subspace_predictions = []
late_subspace_predictions = []

early_subspace = subspaces[0]
late_subspace = subspaces[1]

for run in rundirs:
    source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full = get_subspace_rdms(run, early_subspace)
    early_subspace_predictions.append(source_rdm_sub)
    
for run in rundirs:
    source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full = get_subspace_rdms(run, late_subspace)
    late_subspace_predictions.append(source_rdm_sub)

In [ ]:

# fig, axs = plot_all_rdms_for_subspace_model(source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full)
# plt.suptitle(f"{run} t={subspace[0]}ms, d={subspace[1]}ms")
# plt.tight_layout()

early_subspace_predictions = np.array(early_subspace_predictions)
late_subspace_predictions = np.array(late_subspace_predictions)
print(early_subspace_predictions.shape, late_subspace_predictions.shape)

# avg over folds
avg_early = early_subspace_predictions.mean(axis=1)
avg_late = late_subspace_predictions.mean(axis=1)

early_spearman, _ = spearmanr(avg_early, axis=1)
late_spearman, _ = spearmanr(avg_late, axis=1)

# fill lower with 0 
early_spearman = np.triu(early_spearman)
late_spearman = np.triu(late_spearman)
np.fill_diagonal(early_spearman, 0)
np.fill_diagonal(late_spearman, 0)

early_spearman[early_spearman == 0] = np.nan
late_spearman[late_spearman == 0] = np.nan

early_pairs = squareform(early_spearman, checks=False)
late_pairs = squareform(late_spearman, checks=False)

mean_corr_early, mean_corr_late = early_pairs.mean(), late_pairs.mean()


In [ ]:

plt.figure(figsize=(HALF_WIDTH, HALF_WIDTH / 3))
plt.subplot(131)
plt.imshow(early_spearman, vmin=0, vmax=1)
plt.title(f"early, \nmean rank corr. {mean_corr_early:.2f}")
plt.colorbar(shrink=.5)
plt.gca().axis('off')

plt.subplot(132)
plt.imshow(late_spearman, vmin=0, vmax=1)
plt.title(f"late, \nmean rank corr. {mean_corr_late:.2f}")
plt.colorbar(shrink=.5)
plt.gca().axis('off')

plt.subplot(133)
# all with all
all_rdms = np.concatenate([avg_early, avg_late], axis=0)
all_spearman, _ = spearmanr(all_rdms, axis=1)

plt.title('early vs. late\nrank corr.')
plt.imshow(all_spearman)
plt.colorbar(shrink=.5)
plt.gca().axis('off')


In [ ]:
plt.figure(figsize=(QUARTER_WIDTH * .8, QUARTER_WIDTH * .8))
# all with all
all_rdms = np.concatenate([avg_early, avg_late], axis=0)
all_spearman, _ = spearmanr(all_rdms, axis=1)

# fill diagonal with nan for colormap (diagonal is all 1 by definition)
all_spearman_plot = all_spearman.copy()
np.fill_diagonal(all_spearman_plot, np.nan)

plt.title('Subspace RDMs\nearly vs. late')
plt.imshow(all_spearman_plot)
plt.colorbar(shrink=.5, label=r"spearman's $\rho$")
plt.gca().axis('off')
substr = ""
for t,d in subspaces:
    substr += f"-t-{t}-d-{d}"

fpath = path.join(savedir, f'{run_group_name}{substr}_readout_sub_early_late_rdm_cor.svg')
print(fpath)
plt.savefig(fpath)


## Permutation Test

In [ ]:
def spearman_test_stat(x, y):
    """
    expects two 1d vectors, returns spearman rank order correlation
    """
    assert len(x.shape) == 1, 'first argument must be 1d vector'
    assert len(y.shape) == 1, 'second argument must be 1d vector'
    assert len(x) == len(y), 'vectors are not of same length'
    stat, _ = spearmanr(x, y)
    return stat


def rdm_permutation_test(model, rdm, n_permutations=1_000, test_statistic=spearman_test_stat):
    # make sure we received square matrices
    assert len(model.shape) == 2, f'expected two axes for model rdm, got: {model.shape}'
    assert len(rdm.shape) == 2, f'expected two axes for rdm, got: {rdm.shape}'
    assert model.shape[0] == model.shape[1], 'model is not a square matrix'
    assert rdm.shape[0] == rdm.shape[1], 'rdm is not a square matrix'
    
    model = squareform(model)  # upper triangular

    def shuffle_rdm(rdm):
        """
        jointly shuffles rows and columns of the RDM
        """
        n_entries = len(rdm)
        idx = np.random.permutation(n_entries)
        rdm = rdm.copy()  # safeguard to ensure we don't mess with the original data
        return rdm[idx, :][:, idx]

    true_stat = test_statistic(model, squareform(rdm))
    perm_stats = []
    
    for _ in range(n_permutations):
        rdm_shuff = squareform(shuffle_rdm(rdm))  # shuffle rdm, then take upper triangular
        perm_stats.append(test_statistic(model, rdm_shuff))

    perm_stats = np.array(perm_stats)

    # compute probability of getting a test statistic larger than for the true RDM after shuffling
    prob_larger = np.mean(perm_stats > true_stat)

    # visualize result
    fig = plt.figure(figsize=(FULL_WIDTH, QUARTER_WIDTH))
    plt.subplot(131)
    plt.imshow(squareform(model))
    plt.colorbar(shrink=.7)
    plt.title('model rdm')
    
    plt.subplot(132)
    plt.imshow(rdm)
    plt.colorbar(shrink=.7)
    plt.title('true rdm')
    
    plt.subplot(133)
    plt.hist(perm_stats, bins=50)
    plt.axvline(true_stat, label=f'p={prob_larger}', color='tab:red')
    plt.legend()
    plt.title('permutation test')

    return true_stat, perm_stats, prob_larger, fig


In [ ]:
n_array = len(avg_early)
model = np.ones((2, n_array * 2))
model[0, :n_array] = 0
model[1, n_array:] = 0
model = squareform(pdist(model.T))
print(squareform(model).shape, all_spearman.shape)


# convert spearman correlations to distance matrix
spearman_distances = 1 - all_spearman

# floating point operation causes the diagonal to deviate slightly from zero
# force zeros on the diagonal so squareform does not throw an error
# -- first manually check diagonal is close to zero
assert np.allclose(np.diagonal(spearman_distances), 0), 'distance matrix does not have all-zero diagonal'
# -- now fill diagnonal. This allows squareform to still check whether the matrix is symmetric
np.fill_diagonal(spearman_distances, 0)

true_stat, perm_stats, p_larger, fig = rdm_permutation_test(model, spearman_distances, n_permutations=10_000) 
fpath = path.join(savedir, f'{run_group_name}{substr}_readout_sub_early_late_rdm_cor_permtest.svg')
fig.savefig(fpath)
plt.show()


# Prediction

In [ ]:
early_subspace_predictions = []
late_subspace_predictions = []

early_subspace = subspaces[0]
late_subspace = subspaces[1]

for run in rundirs:
    source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full = get_subspace_rdms(run, early_subspace)
    early_subspace_predictions.append(target_rdm_pred)
    
for run in rundirs:
    source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full = get_subspace_rdms(run, late_subspace)
    late_subspace_predictions.append(target_rdm_pred)

# fig, axs = plot_all_rdms_for_subspace_model(source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full)
# plt.suptitle(f"{run} t={subspace[0]}ms, d={subspace[1]}ms")
# plt.tight_layout()

early_subspace_predictions = np.array(early_subspace_predictions)
late_subspace_predictions = np.array(late_subspace_predictions)

print(early_subspace_predictions.shape, late_subspace_predictions.shape)

# avg over folds
avg_early = early_subspace_predictions.mean(axis=1)
avg_late = late_subspace_predictions.mean(axis=1)

early_spearman, _ = spearmanr(avg_early, axis=1)
late_spearman, _ = spearmanr(avg_late, axis=1)

# fill lower with 0 
early_spearman = np.triu(early_spearman)
late_spearman = np.triu(late_spearman)
np.fill_diagonal(early_spearman, 0)
np.fill_diagonal(late_spearman, 0)

early_spearman[early_spearman == 0] = np.nan
late_spearman[late_spearman == 0] = np.nan

early_pairs = squareform(early_spearman, checks=False)
late_pairs = squareform(late_spearman, checks=False)

mean_corr_early, mean_corr_late = early_pairs.mean(), late_pairs.mean()

plt.figure(figsize=FULL_PANEL_SIZE)
plt.subplot(131)
plt.imshow(early_spearman, vmin=0, vmax=1)
plt.title(f"early, mean rank corr. {mean_corr_early:.2f}")
plt.colorbar(shrink=.3)

plt.subplot(132)
plt.imshow(late_spearman, vmin=0, vmax=1)
plt.title(f"late, mean rank corr. {mean_corr_late:.2f}")
plt.colorbar(shrink=.3)

plt.subplot(133)
# all with all
all_rdms = np.concatenate([avg_early, avg_late], axis=0)
all_spearman, _ = spearmanr(all_rdms, axis=1)

plt.title('RDM rank corr. early vs late pred.')
plt.imshow(all_spearman)
plt.colorbar(shrink=.3)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(QUARTER_WIDTH * .8, QUARTER_WIDTH * .8))
# all with all
all_rdms = np.concatenate([avg_early, avg_late], axis=0)
all_spearman, _ = spearmanr(all_rdms, axis=1)

# fill diagonal with nan for colormap (diagonal is all 1 by definition)
all_spearman_plot = all_spearman.copy()
np.fill_diagonal(all_spearman_plot, np.nan)

plt.title('Predicted RDMs\nearly vs. late')
im = plt.imshow(all_spearman_plot)
plt.colorbar(
    shrink=.5, 
    label=r"spearman's $\rho$", 
    ticks=[
        np.nanmin(all_spearman_plot), # min
        np.nanmax((all_spearman_plot) - np.nanmin(all_spearman_plot)) / 2 + np.nanmin(all_spearman_plot), # midpoint
        np.nanmax(all_spearman_plot) # max
    ],
    format=tkr.FormatStrFormatter('%.2f')
    
)
            
plt.gca().axis('off')
substr = ""
for t,d in subspaces:
    substr += f"-t-{t}-d-{d}"

fpath = path.join(savedir, f'{run_group_name}{substr}_prediction_early_late_rdm_cor.svg')
print(fpath)
plt.savefig(fpath)


In [ ]:
# convert spearman correlations to distance matrix
spearman_distances = 1 - all_spearman

# floating point operation causes the diagonal to deviate slightly from zero
# force zeros on the diagonal so squareform does not throw an error
# -- first manually check diagonal is close to zero
assert np.allclose(np.diagonal(spearman_distances), 0), 'distance matrix does not have all-zero diagonal'
# -- now fill diagnonal. This allows squareform to still check whether the matrix is symmetric
np.fill_diagonal(spearman_distances, 0)

true_stat, perm_stats, p_larger, fig = rdm_permutation_test(model, spearman_distances, n_permutations=10_000) 

fpath = path.join(savedir, f'{run_group_name}{substr}_prediction_early_late_rdm_cor_permtest.svg')
fig.savefig(fpath)
plt.show()


# True Target Signal

In [ ]:
early_subspace_predictions = []
late_subspace_predictions = []

early_subspace = subspaces[0]
late_subspace = subspaces[1]

for run in rundirs:
    source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full = get_subspace_rdms(run, early_subspace)
    early_subspace_predictions.append(target_rdm_full)
    
for run in rundirs:
    source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full = get_subspace_rdms(run, late_subspace)
    late_subspace_predictions.append(target_rdm_full)

# fig, axs = plot_all_rdms_for_subspace_model(source_rdm_full, source_rdm_sub, target_rdm_pred, target_rdm_full)
# plt.suptitle(f"{run} t={subspace[0]}ms, d={subspace[1]}ms")
# plt.tight_layout()



early_subspace_predictions = np.array(early_subspace_predictions)
late_subspace_predictions = np.array(late_subspace_predictions)

print(early_subspace_predictions.shape, late_subspace_predictions.shape)

# avg over folds
avg_early = early_subspace_predictions.mean(axis=1)
avg_late = late_subspace_predictions.mean(axis=1)

early_spearman, _ = spearmanr(avg_early, axis=1)
late_spearman, _ = spearmanr(avg_late, axis=1)

# fill lower with 0 
early_spearman = np.triu(early_spearman)
late_spearman = np.triu(late_spearman)
np.fill_diagonal(early_spearman, 0)
np.fill_diagonal(late_spearman, 0)

early_spearman[early_spearman == 0] = np.nan
late_spearman[late_spearman == 0] = np.nan

early_pairs = squareform(early_spearman, checks=False)
late_pairs = squareform(late_spearman, checks=False)

mean_corr_early, mean_corr_late = early_pairs.mean(), late_pairs.mean()

plt.figure(figsize=FULL_PANEL_SIZE)
plt.subplot(131)
plt.imshow(early_spearman, vmin=0, vmax=1)
plt.title(f"early, mean rank corr. {mean_corr_early:.2f}")
plt.colorbar(shrink=.3)

plt.subplot(132)
plt.imshow(late_spearman, vmin=0, vmax=1)
plt.title(f"late, mean rank corr. {mean_corr_late:.2f}")
plt.colorbar(shrink=.3)

plt.subplot(133)
# all with all
all_rdms = np.concatenate([avg_early, avg_late], axis=0)
all_spearman, _ = spearmanr(all_rdms, axis=1)

plt.title('RDM rank corr. early vs late target signal')
plt.imshow(all_spearman)
plt.colorbar(shrink=.3)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(QUARTER_WIDTH * .8, QUARTER_WIDTH * .8))
# all with all
all_rdms = np.concatenate([avg_early, avg_late], axis=0)
all_spearman, _ = spearmanr(all_rdms, axis=1)

plt.title('Subspace RDMs\nearly vs. late')
plt.imshow(all_spearman)
plt.colorbar(shrink=.5, label=r"spearman's $\rho$")
plt.gca().axis('off')
substr = ""
for t,d in subspaces:
    substr += f"-t-{t}-d-{d}"

fpath = path.join(savedir, f'{run_group_name}{substr}_target_early_late_rdm_cor.svg')
print(fpath)
plt.savefig(fpath)


In [ ]:
# convert spearman correlations to distance matrix
spearman_distances = 1 - all_spearman

# floating point operation causes the diagonal to deviate slightly from zero
# force zeros on the diagonal so squareform does not throw an error
# -- first manually check diagonal is close to zero
assert np.allclose(np.diagonal(spearman_distances), 0), 'distance matrix does not have all-zero diagonal'
# -- now fill diagnonal. This allows squareform to still check whether the matrix is symmetric
np.fill_diagonal(spearman_distances, 0)

true_stat, perm_stats, p_larger, fig = rdm_permutation_test(model, spearman_distances, n_permutations=10_000) 


fpath = path.join(savedir, f'{run_group_name}{substr}_target_early_late_rdm_cor_permtest.svg')
fig.savefig(fpath)
plt.show()


# Unique Variance Explained by Model RDMs in either Subspace

In [ ]:
from ephyslib.rdm import unique_variance_per_model
from tqdm import tqdm
import sys
from joblib import Parallel, delayed


keys_select = [
    f"layer{i}.1.conv2" for i in range(1, 5)
]
keys_select = ['conv1'] + keys_select


model_rdm_folder = path.join("..", "..", "datasets", "TIMM", "resnet18")
model_rdm_name = "rdms-resnet18-metric_cosine-normalization_None-pca_1000.pkl"

with open(path.join(model_rdm_folder, model_rdm_name), "rb") as f:
    model_rdm_data = pickle.load(f)

model_keys = model_rdm_data["selected_nodes"]
model_rdm_dict = model_rdm_data["rdms"]


In [ ]:
def get_unique_variance_trajectory(rundir, subspace):
    
    # get rdms
    analysis_dir = path.join("..", "..", "results", "inter_area", rundir, "analysis")
    tsub, dsub = subspace
    subdir = path.join(analysis_dir, f"subspace_t-{tsub}_d-{dsub}")
    
    # ----- collect source data dnn corrs
    # get source timecourse data first, same for all subspaces so pick from first folder
    with open(path.join(subdir, 'rdms.pkl'), 'rb') as f:
        sub_data = pickle.load(f)
    print(sub_data.keys())
    
    time = sub_data['time']
    rdms = sub_data['rdms_source_readout_subspace'].mean(axis=0)  # mean over folds
    print(rdms.shape)
    
    models = np.array([model_rdm_dict[key] for key in keys_select])
    print(models.shape)
    
    full_vars_exp = []
    partial_vars_exp = []
    unique_vars_exp = []
    
    with Parallel(return_as='generator', n_jobs=-1, verbose=0) as pool:
        res_gen = pool(delayed(unique_variance_per_model)(models, rdm, True, False, True) for rdm in rdms)
        
        for res in res_gen:
            f, p, u = res
            full_vars_exp.append(f)
            partial_vars_exp.append(p)
            unique_vars_exp.append(u)
            
    unique_vars_exp = np.array(unique_vars_exp)
    full_vars_exp = np.array(full_vars_exp)
    print(unique_vars_exp.shape, full_vars_exp.shape)

    return time, unique_vars_exp, full_vars_exp

unique_vars_early, full_vars_early = [], []
unique_vars_late, full_vars_late = [], []

for rundir in rundirs:
    t, u, f = get_unique_variance_trajectory(rundir, subspaces[0])
    unique_vars_early.append(u)
    full_vars_early.append(f)

for rundir in rundirs:
    t, u, f = get_unique_variance_trajectory(rundir, subspaces[1])
    unique_vars_late.append(u)
    full_vars_late.append(f)

full_vars_early = np.array(full_vars_early)
full_vars_late = np.array(full_vars_late)

unique_vars_early = np.array(unique_vars_early)
unique_vars_late = np.array(unique_vars_late)


In [ ]:
ylims = (0, .13)


unique_vars_exp_early_avg = unique_vars_early.mean(axis=0)
unique_vars_exp_late_avg = unique_vars_late.mean(axis=0)

full_vars_early_avg = full_vars_early.mean(axis=0)
full_vars_late_avg = full_vars_late.mean(axis=0)

plt.figure(figsize=(FULL_PANEL_SIZE[0], 2))
plt.subplot(121)
for key, unique_exp in zip(keys_select, unique_vars_exp_early_avg.T):
    plt.plot(time, unique_exp, label=key)
plt.plot(time, full_vars_early_avg, color='k', label='all included models')
plt.xlabel('time (ms)')
plt.ylabel('unique explained variance')
plt.legend()
plt.title('early')
plt.ylim(ylims)
plt.subplot(122)
for key, unique_exp in zip(keys_select, unique_vars_exp_late_avg.T):
    plt.plot(time, unique_exp, label=key)
plt.plot(time, full_vars_late_avg, color='k', label='all included models')
plt.xlabel('time (ms)')
plt.ylabel('unique explained variance')
plt.legend()
plt.title('late')
plt.ylim(ylims)
plt.tight_layout()

# Unique Variance Trajectories - early vs late

Show unique variance in target RDM explained by either RDMs predicted from the early or late model.

In [ ]:
from joblib import Parallel, delayed

from ephyslib.rdm import unique_variance_per_model_with_permtest

def unique_variance_by_subspace_rdms(run, subspaces, ts_select):
    # get rdms
    analysis_dir = path.join("..", "..", "results", "inter_area", run, "analysis")
    subspace_rdm_trajectories = []
    for i, subspace in enumerate(subspaces):
        tsub, dsub = subspace
        subdir = path.join(analysis_dir, f"subspace_t-{tsub}_d-{dsub}")
        
        # ----- collect source data dnn corrs
        # get source timecourse data first, same for all subspaces so pick from first folder
        with open(path.join(subdir, 'rdms.pkl'), 'rb') as f:
            sub_data = pickle.load(f)
        print(sub_data.keys())
        
        # target data is the same for all subspaces, only retrieve once
        if i == 0:
            rdms_target = sub_data['rdms_target']
            time = sub_data['time']
        sub_rdms = sub_data['rdms_target_prediction']
        subspace_rdm_trajectories.append(sub_rdms)

    subspace_rdm_trajectories = np.array(subspace_rdm_trajectories)
    print(rdms_target.shape)
    print(subspace_rdm_trajectories.shape)

    # avg. over folds
    # --- target
    rdms_target = rdms_target.mean(axis=0)
    # --- predictors
    subspace_rdm_trajectories = subspace_rdm_trajectories.mean(axis=1)
    
    print(f"target rdms: {rdms_target.shape}")
    print(f"predictors: {subspace_rdm_trajectories.shape}")

    # --- iterate over time

    full_vars_per_t, unique_vars_per_t = [], []
    
    for t in tqdm(ts_select, file=sys.stdout):
        # make sure time is aligned: predictors need to be shifted by their respective delay
        t_idx_target = np.where(time == t)[0][0]
        t_idx_subs = []
        for subspace in subspaces:
            tsub, dsub = subspace
            t_idx_sub = np.where((time + dsub) == t)[0][0]
            t_idx_subs.append(t_idx_sub)

        # --- select target at time t
        rdm_t_target = rdms_target[t_idx_target]

        # --- get predictors at time t - d
        predictors = []
        for srt, idx in zip(subspace_rdm_trajectories, t_idx_subs):
            predictors.append(srt[idx])

        # --- build model
        models = np.stack(predictors)
        
        full_var, _, unique_vars = unique_variance_per_model(models, rdm_t_target, zscore=True, fit_intercept=False, positive=True) 
        full_vars_per_t.append(full_var)
        unique_vars_per_t.append(unique_vars)

    return np.array(full_vars_per_t), np.array(unique_vars_per_t)



def unique_variance_by_subspace_rdms_with_permtest(run, subspaces, ts_select, perms):
    # get rdms
    analysis_dir = path.join("..", "..", "results", "inter_area", run, "analysis")
    subspace_rdm_trajectories = []
    for i, subspace in enumerate(subspaces):
        tsub, dsub = subspace
        subdir = path.join(analysis_dir, f"subspace_t-{tsub}_d-{dsub}")
        
        # ----- collect source data dnn corrs
        # get source timecourse data first, same for all subspaces so pick from first folder
        with open(path.join(subdir, 'rdms.pkl'), 'rb') as f:
            sub_data = pickle.load(f)
        print(sub_data.keys())
        
        # target data is the same for all subspaces, only retrieve once
        if i == 0:
            rdms_target = sub_data['rdms_target']
            time = sub_data['time']
        sub_rdms = sub_data['rdms_target_prediction']
        subspace_rdm_trajectories.append(sub_rdms)

    subspace_rdm_trajectories = np.array(subspace_rdm_trajectories)

    # avg. over folds
    # --- target
    rdms_target = rdms_target.mean(axis=0)
    # --- predictors
    subspace_rdm_trajectories = subspace_rdm_trajectories.mean(axis=1)
    
    print(f"target rdms: {rdms_target.shape}")
    print(f"predictors: {subspace_rdm_trajectories.shape}")


    # ----------------------
    #   iterate over time
    # ----------------------

    # --- set up function to call for each t
    def compute_for_t(t):
        print(f"time: {t}ms")
        # make sure time is aligned: predictors need to be shifted by their respective delay
        t_idx_target = np.where(time == t)[0][0]
        t_idx_subs = []
        
        # -----------------------------
        #    iterate over predictors
        # -----------------------------
        
        for subspace in subspaces:
            # predictions are time lagged, get predicted RDMs shifted by delay of model fit
            tsub, dsub = subspace
            t_idx_sub = np.where((time + dsub) == t)[0][0]
            t_idx_subs.append(t_idx_sub)

        # --- select target at time t
        rdm_t_target = rdms_target[t_idx_target]

        # --- get predictors at time t - d
        predictors = []
        for srt, idx in zip(subspace_rdm_trajectories, t_idx_subs):
            predictors.append(srt[idx])

        # --- build model
        models = np.stack(predictors)
        
        # full_var, partial_vars, unique_vars, shuff_full_dists, shuff_unique_dists = unique_variance_per_model_with_permtest(models, rdm_t_target, perms, zscore=True, fit_intercept=False, positive=True) 
        return unique_variance_per_model_with_permtest(models, rdm_t_target, perms, zscore=True, fit_intercept=False, positive=True) 
        

    full_vars_per_t, partial_vars_per_t, unique_vars_per_t = [], [], []
    shuffle_full_vars_per_t, shuffle_unique_vars_per_t = [], []  # keep track of explained variance for shuffled predictors

    # create pool context
    with Parallel(n_jobs=-1, return_as='generator', verbose=0) as pool:
        # --- submit
        func = delayed(compute_for_t)
        res_gen = pool(func(t) for t in ts_select)

        # --- collect
        for full_var, partial_vars, unique_vars, shuff_full_dists, shuff_unique_dists in tqdm(res_gen):
            full_vars_per_t.append(full_var)
            unique_vars_per_t.append(unique_vars)
            partial_vars_per_t.append(partial_vars)
            shuffle_full_vars_per_t.append(shuff_full_dists)
            shuffle_unique_vars_per_t.append(shuff_unique_dists)

    return np.array(full_vars_per_t), np.array(unique_vars_per_t), np.array(partial_vars_per_t), np.array(shuffle_full_vars_per_t), np.array(shuffle_unique_vars_per_t)


In [ ]:
# --- compute for all runs

compute_permtest = False
n_entries_rdm = 1_000  # number of conditions in the RDM
n_perms = 1_000        # number of permutations to test against

# generate permutations - these must be the same for all conditions + arrays so we can average later
permutations = np.array([
    np.random.permutation(n_entries_rdm) for _ in range(n_perms)
])

ts_select = np.arange(0, 300, 2)
full_per_run, unique_per_run, shuffle_unique_per_run = [], [], []

for run in rundirs:

    if compute_permtest:
        full_vars, unique_vars, partial_vars, shuffle_full_vars, shuffle_unique_vars = unique_variance_by_subspace_rdms_with_permtest(run, subspaces, ts_select, permutations)
        full_per_run.append(full_vars)
        unique_per_run.append(unique_vars)
        shuffle_unique_per_run.append(shuffle_unique_vars)
    else:
        full_vars, unique_vars = unique_variance_by_subspace_rdms(run, subspaces, ts_select)
        full_per_run.append(full_vars)
        unique_per_run.append(unique_vars)
    
    plt.figure(figsize=THIRD_PANEL_SIZE)
    plt.plot(ts_select, full_vars, color='k', label='full model')
    plt.plot(ts_select, unique_vars)
    plt.xlabel('time (ms)')
    plt.ylabel('unique exp. var.')
    plt.title(run)
    
    for subspace in subspaces:
        plt.axvline(subspace[0], color='r', linewidth=.5)

    fpath = path.join(savedir, run + "_early_late_sub_rdm_unique_expvar.svg")
    plt.savefig(fpath)

In [ ]:
full_per_run = np.array(full_per_run)
unique_per_run = np.array(unique_per_run)

print(full_per_run.shape, unique_per_run.shape)

plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH/2))
for subspace in subspaces:
    plt.axvline(subspace[0], color='k', linewidth=.5, alpha=.3)
plt.plot(ts_select, full_per_run.mean(axis=0), color='k', label='full model')
plt.plot(ts_select, unique_per_run.mean(axis=0))

plt.xlabel('time (ms)')
plt.ylabel('unique exp. var.')
plt.title(run_group_name)
fpath = path.join(savedir, run_group_name + "_early_late_sub_rdm_unique_expvar.svg")
plt.savefig(fpath)

In [ ]:
full_per_run = np.array(full_per_run)
unique_per_run = np.array(unique_per_run)

print(full_per_run.shape, unique_per_run.shape)



In [ ]:
plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH/2))
for subspace in subspaces:
    plt.axvline(subspace[0], color='k', linewidth=.5, alpha=.3)
for i in range(unique_per_run.shape[-1]):
    plt.plot(ts_select, unique_per_run[..., 0].T, color='tab:blue', linewidth=.3)
    plt.plot(ts_select, unique_per_run[..., 1].T, color='tab:orange', linewidth=.3)

plt.xlabel('time (ms)')
plt.ylabel('unique exp. var.')

In [ ]:
full_per_run.shape

In [ ]:
import mne

norm_expvars = True 

group1, group2 = unique_per_run[:, :, 0], unique_per_run[:, :, 1]
if norm_expvars:
    group1 = group1 / full_per_run
    group2 = group2 / full_per_run
    
print(group1.shape, group2.shape)

stat_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_test(
    [group1, group2], 
)


In [ ]:
plt.figure(figsize=(FULL_WIDTH, FULL_WIDTH / 4))
plt.subplot(131)
plt.plot(ts_select, stat_obs)
plt.ylabel('test statistic')
plt.xlabel('time (ms)')

# plt.subplot(142)
# plt.plot(ts_select, uncorrected_ps, label='uncorrected p-vals')
# plt.plot(ts_select, cluster_pv, label='tfce p-vals')
# plt.axhline(0.05, color='r', linewidth=0.2)
# plt.legend(loc='lower right')

plt.subplot(132)
for subspace in subspaces:
    plt.axvline(subspace[0], color='k', linewidth=.5, alpha=.3)
plt.plot(ts_select, full_per_run.mean(axis=0), color='k', label='full model')
plt.plot(ts_select, unique_per_run.mean(axis=0))

for cluster, pv in zip(clusters, cluster_pv):
    if pv < 0.05:
        print(pv)
        plt.scatter(ts_select[cluster], np.ones_like(cluster) * -.01, s=1, color='tab:red')

plt.xlabel('times (ms)')
plt.ylabel('unique explained variance')

plt.subplot(133)
plt.hist(H0, bins=50)


In [ ]:
clusters = [np.array(cluster).squeeze() for cluster in clusters]

for cluster in clusters:
    assert (np.diff(cluster) == 1).all(), 'clusters have missing indices. Something went wrong'
    
cluster_intervals = [
    (ts_select[c[0]], ts_select[c[-1]]) for c in clusters
]

for i, (p, cinter) in enumerate(zip(cluster_pv, cluster_intervals)):
    print(f"cluster {i+1}: {cinter[0]}-{cinter[1]}ms, p-val: {p:.3f}")


In [ ]:
xlims = (0, 300)
plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH/2))
for subspace in subspaces:
    plt.axvline(subspace[0], color='k', linewidth=.5, alpha=.3)
plt.plot(ts_select, full_per_run.mean(axis=0), color='k')
plt.plot(ts_select, unique_per_run.mean(axis=0))

for extent, pv in zip(cluster_intervals, cluster_pv):
    if pv < 0.05:
        plt.plot(extent, np.ones_like(extent) * -0.01, label=f"{extent[0]}-{extent[1]}ms, p-val: {pv:.3f}")

plt.xlabel('time (ms)')
plt.ylabel('unique exp. var.')
plt.xlim(xlims)
plt.title(run_group_name)
plt.legend()
fpath = path.join(savedir, run_group_name + "_early_late_sub_rdm_unique_expvar_clustperm.svg")
plt.savefig(fpath)